In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("dataOriginal") if Path("dataOriginal").is_dir() else Path("../dataOriginal")
assert data_dir.is_dir(), "Ejecuta desde la raíz del proyecto o notebooks/"

for path in sorted(data_dir.glob("*.csv")):
    muestra = pd.read_csv(path, nrows=5)
    print(f"\n--- {path.name} ---")
    print("Tamaño:", f"{path.stat().st_size / 1_000_000:.1f} MB")
    display(muestra)

In [ ]:
for nombre in ["stores.csv", "holidays_events.csv", "oil.csv", "transactions.csv"]:
    df = pd.read_csv(data_dir / nombre)

    print(f"\n--- {nombre} ---")
    print("Filas y columnas:", df.shape)
    print("Duplicados exactos:", df.duplicated().sum())

    if "date" in df.columns:
        fechas = pd.to_datetime(df["date"], errors="coerce")
        print("Rango de fechas:", fechas.min(), "a", fechas.max())
        print("Fechas que no se pudieron leer:", fechas.isna().sum())

    faltantes = df.isna().sum()
    print("Valores vacíos por columna:")
    print(faltantes[faltantes > 0].to_string() if faltantes.any() else "Ninguno")

In [ ]:
from collections import Counter

train_path = data_dir / "train.csv"

total_filas = 0
faltantes = Counter()
tiendas = set()
familias = set()
min_fecha = None
max_fecha = None
min_ventas = None
max_ventas = None
suma_ventas = 0
filas_con_venta_cero = 0
ventas_negativas = 0

for bloque in pd.read_csv(train_path, chunksize=250_000):
    bloque["date"] = pd.to_datetime(bloque["date"], errors="coerce")

    total_filas += len(bloque)
    faltantes.update(bloque.isna().sum().to_dict())
    tiendas.update(bloque["store_nbr"].dropna().unique())
    familias.update(bloque["family"].dropna().unique())

    fecha_min_bloque = bloque["date"].min()
    fecha_max_bloque = bloque["date"].max()
    min_fecha = fecha_min_bloque if min_fecha is None else min(min_fecha, fecha_min_bloque)
    max_fecha = fecha_max_bloque if max_fecha is None else max(max_fecha, fecha_max_bloque)

    min_bloque = bloque["sales"].min()
    max_bloque = bloque["sales"].max()
    min_ventas = min_bloque if min_ventas is None else min(min_ventas, min_bloque)
    max_ventas = max_bloque if max_ventas is None else max(max_ventas, max_bloque)

    suma_ventas += bloque["sales"].sum()
    filas_con_venta_cero += (bloque["sales"] == 0).sum()
    ventas_negativas += (bloque["sales"] < 0).sum()

print("Filas:", f"{total_filas:,}")
print("Fechas:", min_fecha, "a", max_fecha)
print("Tiendas:", len(tiendas), "| Familias:", len(familias))
print("Ventas: mínimo", min_ventas, "| promedio", round(suma_ventas / total_filas, 2), "| máximo", max_ventas)
print("Filas con venta cero:", f"{filas_con_venta_cero:,}")
print("Ventas negativas:", f"{ventas_negativas:,}")
print("Valores vacíos:", {col: n for col, n in faltantes.items() if n > 0})

In [ ]:
partes = []
mayores = []

for bloque in pd.read_csv(
    data_dir / "train.csv",
    usecols=["date", "store_nbr", "family", "sales"],
    chunksize=250_000,
):
    resumen = bloque.groupby("family").agg(
        filas=("sales", "size"),
        ceros=("sales", lambda ventas: (ventas == 0).sum()),
        ventas_totales=("sales", "sum"),
        venta_maxima=("sales", "max"),
    )
    partes.append(resumen)
    mayores.append(bloque.nlargest(10, "sales"))

# Al unir bloques: sumar conteos y ventas, pero tomar el máximo de máximos.
resumen_familias = pd.concat(partes).groupby(level=0).agg(
    filas=("filas", "sum"),
    ceros=("ceros", "sum"),
    ventas_totales=("ventas_totales", "sum"),
    venta_maxima=("venta_maxima", "max"),
)
resumen_familias["porcentaje_ceros"] = (
    resumen_familias["ceros"] / resumen_familias["filas"] * 100
)
resumen_familias["venta_promedio"] = (
    resumen_familias["ventas_totales"] / resumen_familias["filas"]
)

display(
    resumen_familias[
        ["filas", "porcentaje_ceros", "venta_promedio", "venta_maxima"]
    ].sort_values("porcentaje_ceros", ascending=False).round(2)
)

mayores_ventas = pd.concat(mayores).nlargest(10, "sales")
display(mayores_ventas)